# 📈 04. Avaliação no Teste, Matriz de Custo & Explicabilidade SHAP
### **Tech Challenge - Fase 3: Predição de Alfabetização Infantil**

Este notebook aborda o quarto passo do ciclo de Machine Learning:
1. **Avaliação rigorosa no Conjunto de Teste Independente** (ROC-AUC, PR-AUC, F1-Score, Acurácia, Brier Score);
2. **Matriz de Custo Educacional & Otimização do Limiar de Decisão (*Threshold Tuning*)** para maximizar o Recall de alunos vulneráveis;
3. **Explicabilidade de Inteligência Artificial (XAI) com SHAP** (Beeswarm, Feature Importance Global, Decomposição por Pilares e Waterfall Individual);
4. **Recomendações Práticas para Políticas Públicas**.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import RANDOM_STATE, MODELS_DIR
from src.data_loader import load_gold_silver_data
from src.preprocessing import split_and_preprocess_data
from src.models import get_candidate_models, fit_and_save_all_models
from src.tuning import tune_lightgbm_optuna
from src.evaluation import evaluate_all_models_on_test, optimize_decision_threshold
from src.explainability import explain_model_with_shap
import joblib

sns.set_theme(style='whitegrid', palette='deep')
%matplotlib inline
print('Configuração pronta!')

## 1. Carregamento de Dados e Modelos

In [ ]:
df_raw = load_gold_silver_data(sample_size=30000, seed=RANDOM_STATE)
X_train, X_test, y_train, y_test, feature_names, preprocessor = split_and_preprocess_data(df_raw)

# Carregar modelos persistidos em models_saved/
trained_models = {}
for model_file in MODELS_DIR.glob('*.joblib'):
    name = model_file.stem.replace('_', ' ').title().replace(' ', '_')
    trained_models[name] = joblib.load(model_file)

print(f'Modelos carregados para teste: {list(trained_models.keys())}')

## 2. Avaliação de Desempenho no Teste Independente (6.000 amostras)

In [ ]:
df_test_metrics = evaluate_all_models_on_test(trained_models, X_test, y_test, save_figures=True)
display(df_test_metrics)

## 3. Matriz de Decisão Social & Otimização do Limiar (*Threshold Tuning*)
Ajustando o corte probabilístico para priorizar a proteção e captura de crianças em risco.

In [ ]:
best_model = trained_models.get('Lightgbm_Optimized', list(trained_models.values())[0])
thresh_res = optimize_decision_threshold(best_model, X_test, y_test, save_figures=True)

## 4. Explicabilidade do Modelo com SHAP (Explainable AI)

In [ ]:
shap_results = explain_model_with_shap(best_model, X_test, feature_names, sample_size=1500, save_figures=True)